# R14 — ICLR fully matched persona × target-breadth factorial

R12의 검증된 Kaggle 설치·학습·7렌즈 평가 셸을 재사용한다. BATCH 0은 동일한 새
instruction catalog에서 arm1/2/4/5 응답을 대칭 생성하고, BATCH 1–5는 그 고정 ZIP만
복원해 12개 학습 셀을 실행한다. 세션에서 바꾸는 값은 `BATCH` 하나뿐이다.


## 0. Secrets

⚠️ **Secrets는 계정이 아니라 노트북마다 붙는다.** 새 노트북을 만들거나 업로드하면 계정에 등록돼
있어도 안 딸려온다 → **Add-ons → Secrets → `HF_TOKEN` 토글 ON.**

In [ ]:
import os, torch

# 가속기는 시크릿과 성질이 같다 — **노트북마다** 붙고, Import하면 None으로 초기화된다.
# 여기서 안 막으면 패키지 설치를 다 하고 unsloth 내부 스택에서 죽는다.
if not torch.cuda.is_available():
    raise SystemExit(
        "\n[GPU 없음] 이 세션에 가속기가 안 붙었다. unsloth는 CPU에서 로드 자체가 안 된다.\n"
        "  1) 우측 Settings -> Accelerator -> GPU T4 x2 (또는 P100) 선택 후 다시 Run.\n"
        "     노트북을 새로 Import하면 가속기 설정은 따라오지 않는다.\n"
        "  2) 이미 GPU로 돼 있다면 **주간 할당량(30h) 소진**이다 — 우측 상단 게이지를 볼 것.\n"
        "     소진되면 Kaggle은 경고 없이 CPU로 돌린다.\n"
        f"  torch={torch.__version__} cuda_built={torch.version.cuda}"
    )
print(f"GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}")

from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        "\n[HF_TOKEN 없음] Secrets는 계정이 아니라 **노트북마다** 붙여야 한다.\n"
        "  상단 메뉴 Add-ons -> Secrets -> HF_TOKEN 토글 ON -> 다시 Run\n"
        "  (계정에 이미 등록돼 있어도 새 노트북에는 자동으로 안 딸려온다)\n"
        f"  원본 오류: {e}"
    )
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 1. 패키지 설치 (매 세션 필수)
비교 가능성 핀: `unsloth==2026.7.6`, `transformers==4.56.2`, `trl==0.22.2`.

In [ ]:
import subprocess, sys

def pip(*args):
    """`!pip` 매직 대신 subprocess — 이 노트북의 모든 셀이 순수 파이썬이라
    업로드 전에 compile()로 문법 검증을 걸 수 있다(RUNBOOK 함정 15)."""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("unsloth==2026.7.6")
pip("transformers==4.56.2", "trl==0.22.2")

from importlib.metadata import version
_installed_unsloth = version("unsloth")
assert _installed_unsloth == "2026.7.6", (
    f"Unsloth 설치 핀 불일치: 기대=2026.7.6 실제={_installed_unsloth}")
pip("git+https://github.com/dsbowen/strong_reject.git@main")
pip("lm-eval")
print("packages ok")

## 2. 실험 폴더 + 학습 데이터 복구
데이터는 1차 것을 **그대로 재사용**한다 — 다시 만들면 1차와의 비교가 끊긴다.

In [ ]:
import os, shutil, sys, glob

EXP_ROOT = "/kaggle/working/experiment"
CKPT_ROOT = "/kaggle/tmp/ckpt"

def _input_tree():
    return sorted(glob.glob("/kaggle/input/*")
                  + glob.glob("/kaggle/input/*/*")
                  + glob.glob("/kaggle/input/*/*/*"))

if not os.path.exists(EXP_ROOT):
    hits = glob.glob("/kaggle/input/**/src/eval_refusal.py", recursive=True)
    if not hits:
        print("\n".join(_input_tree()))
        raise SystemExit("branch-1-premise code Dataset을 이 노트북에 붙이세요")
    input_dir = os.path.dirname(os.path.dirname(hits[0]))
    shutil.copytree(input_dir, EXP_ROOT)
os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(EXP_ROOT)
sys.path.insert(0, os.path.join(EXP_ROOT, "src"))

for path, needle in [
    ("src/gen_r14_matched_data.py", "r14_matched_factorial_data_v1"),
    ("src/train_round.py", "save_adapter_dir"),
    ("src/eval_refusal.py", "out_suffix"),
    ("configs/arms.json", '"nb_r14": "R14a"'),
]:
    assert os.path.exists(path), f"R14 code Dataset 아님: {path} 없음"
    assert needle in open(path, encoding="utf-8").read(), f"R14 code Dataset이 낡음: {path}"
print("[gate] R14 code payload 확인")


## 3. 세션 함수 정의

In [ ]:
import json, shutil, subprocess, os

BASE = "unsloth/Meta-Llama-3.1-8B-Instruct"
FULL = " --full"       # R2는 전 라운드 n=313. 1차의 n 불일치가 라운드 비교를 막았다.
MMLU_LIMIT = "30"

def run(cmd: str):
    print(f"\n$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True)
    assert r.returncode == 0, f"실패: {cmd}"

def check(path, min_kb=20):
    """per-prompt 레코드가 실제로 쓰였는지 즉시 확인 — 1차의 핵심 실패를 막는 게이트."""
    kb = os.path.getsize(path) / 1024
    print(f"[check] {os.path.basename(path)} = {kb:.0f} KB")
    assert kb > min_kb, f"per-prompt 레코드가 비었다: {path}"

def probe_baseline(seed, harm_frame="judgement"):
    """동결기저(θ_0, ∅). 학습 전에 한 번. 이후 프로브가 이걸 기준으로 읽는다.
       ⚠️기저와 이후 라운드의 harm_frame이 다르면 ρ·D가 무의미해진다 — 같은 값을 쓸 것."""
    run(f"python src/probe_directions.py --model_path {BASE} --arm arm0 --round 0 --seed {seed} "
        f"--harm_frame {harm_frame}")
    return f"results/probe_arm0_r0_s{seed}.json"

def run_session(arm, seed, lenses="none,agreeable,principled",
                probe=False, frac=False, mmlu=True, eval_rounds=(1, 2, 3),
                harm_frame="judgement", mmlu_limit=None, free_ckpt=True,
                probe_rounds=None, max_new_tokens=256, out_suffix="", save_adapters=""):
    """★2026-07-29 확장 3건:
       eval_rounds  — 학습은 순차라 건너뛸 수 없지만 **평가는 건너뛸 수 있다.**
                      비용의 거의 전부가 eval(1렌즈 n=313 = ~1,800s)이므로 라운드3만
                      재면 세션이 ~3.4h 짧아진다. 시드 반복은 라운드별 곡선이 아니라
                      최종 효과만 필요하므로 eval_rounds=(3,)이 맞다.
       harm_frame   — 프로브 v_harm 추출 프레이밍. 기본 judgement(고친 것).
                      raw는 S1·S2 재현 전용이며 κ_harm≡1이라 무정보다.
       mmlu_limit   — None이면 전역 MMLU_LIMIT. 논문이 보고하는 대조는 **동일 570문항
                      (limit=10)** 이므로 새 arm은 10으로 도는 편이 비교 가능하고 3배 싸다.
       probe_rounds — ★R4 신설. 프로브가 `r in eval_rounds`에 묶여 있어서
                      **eval 없이 프로브만** 뜰 수 없었다. 기하의 라운드 궤적을 재려면
                      eval(렌즈당 25분)을 다시 돌릴 이유가 없다 — 시드42 eval은 이미 있다.
                      None이면 eval_rounds를 따른다(기존 동작 그대로).
       max_new_tokens, out_suffix — ★R5 신설. 512-token 평가는 기존 256-token
                      결과와 파일명이 절대 겹치지 않게 접미사를 함께 쓴다."""
    base_dirs = probe_baseline(seed, harm_frame) if probe else None
    pr_rounds = eval_rounds if probe_rounds is None else probe_rounds
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    prev, prev_dir = BASE, None

    for r in [1, 2, 3]:
        ckpt = f"{CKPT_ROOT}/{arm}_r{r}_s{seed}"
        frac_arg = " --frac_ckpt 0.25,0.5" if (frac and r == 1) else ""
        run(f"python src/train_round.py --arm {arm} --round {r} --seed {seed} "
            f"--base_model {prev} --data data/{arm}/round{r}.jsonl "
            f"--output_dir {ckpt}{frac_arg}"
            + (f" --save_adapter_dir {save_adapters}/{arm}_r{r}_s{seed}" if save_adapters else ""))

        # ★R5: 라운드 r 머지본이 생긴 순간 r-1은 죽은 무게다. 원래는 eval/probe/mmlu가
        #   끝난 뒤에야 지웠는데, 4렌즈 eval은 약 2시간이라 그동안 32GB를 붙들고 있었다.
        #   R5 배치 2가 실제로 여기서 디스크를 터뜨렸다(Docker daemon 오류는 그 증상).
        if prev_dir and os.path.exists(prev_dir):
            shutil.rmtree(prev_dir)
            print(f"[disk] 삭제(학습 직후): {prev_dir}")
            prev_dir = None

        # 세분 라운드: 라운드1 학습 중 저장된 어댑터를 원본 BASE 위에 얹어 평가
        if frac and r == 1:
            for f in ["0p25", "0p5"]:
                ad = f"{CKPT_ROOT}/{arm}_s{seed}/adapter_frac{f}"
                if not os.path.exists(ad):
                    print(f"[warn] 세분 체크포인트 없음: {ad}")
                    continue
                rn = "0.25" if f == "0p25" else "0.5"
                run(f"python src/eval_refusal.py --model_path {BASE} --adapter_path {ad} "
                    f"--arm {arm} --round {rn} --seed {seed} --lenses {lenses}{FULL}{eval_args}")
                check(f"results/eval_{arm}_r{f}_s{seed}{out_suffix}.json")
                if probe:
                    run(f"python src/probe_directions.py --model_path {BASE} --adapter_path {ad} "
                        f"--arm {arm} --round {rn} --seed {seed} --baseline_dirs {base_dirs}")

        if r in eval_rounds:
            run(f"python src/eval_refusal.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
            check(f"results/eval_{arm}_r{r}_s{seed}{out_suffix}.json")
        else:
            print(f"[skip] eval arm={arm} r={r} — eval_rounds={eval_rounds} (비용 절감)")

        if probe and r in pr_rounds:
            run(f"python src/probe_directions.py --model_path {ckpt} --arm {arm} --round {r} "
                f"--seed {seed} --baseline_dirs {base_dirs} --harm_frame {harm_frame}")

        if mmlu and r == 3:
            run(f"bash src/run_mmlu.sh {ckpt} {arm} {r} {mmlu_limit or MMLU_LIMIT}")

        # 디스크 해제는 학습 직후로 옮겼다(R5). 여기서는 다음 라운드 기준만 갱신한다.
        prev, prev_dir = ckpt, ckpt

    # 루프는 '이전' 라운드만 지운다 — 마지막 라운드 머지본 16GB가 그대로 남는다.
    # 한 세션만 돌면 무해하지만, 세션을 이어 돌리면 세션당 16GB씩 쌓여
    # 시간이 아니라 디스크가 먼저 터진다. MMLU는 위 루프 안에서 이미 끝났다.
    if free_ckpt and prev_dir and os.path.exists(prev_dir):
        shutil.rmtree(prev_dir)
        print(f"[disk] 삭제(최종): {prev_dir}")
        return None
    return prev

def run_baseline(seed, lenses="none,agreeable,principled", mmlu_limit=None,
                 mmlu=True, max_new_tokens=256, out_suffix=""):
    """arm0 = 원본 모델. 학습 없음. 페르소나 방아쇠 자체가 무해함을 보이는 통제군.
       mmlu — ★R4 신설. arm0의 limit=10 MMLU는 이미 있다. 다시 돌리면 20분을 버리고
              results/mmlu_arm0_r0/ 에 타임스탬프 파일이 하나 더 쌓인다.
       max_new_tokens, out_suffix — ★R5 신설. 생성 상한과 충돌 없는 파일 접미사."""
    assert max_new_tokens > 0
    assert not out_suffix or (out_suffix.startswith("_") and out_suffix[1:].replace("_", "").replace("-", "").isalnum())
    eval_args = f" --max_new_tokens {max_new_tokens}" + (f" --out_suffix {out_suffix}" if out_suffix else "")
    run(f"python src/eval_refusal.py --model_path {BASE} --arm arm0 --round 0 "
        f"--seed {seed} --lenses {lenses}{FULL}{eval_args}")
    check(f"results/eval_arm0_r0_s{seed}{out_suffix}.json")
    if mmlu:
        run(f"bash src/run_mmlu.sh {BASE} arm0 0 {mmlu_limit or MMLU_LIMIT}")
    else:
        print("[skip] MMLU arm0 — 이미 측정됨(limit=10)")

print("정의 완료")

## 4. 이번 세션 — `BATCH` 하나만 바꾼다

- `0`: 공통 데이터 생성 전용. 결과 ZIP을 회수·수동검토한 뒤 Kaggle Dataset으로 올린다.
- `1`–`5`: 고정 데이터 ZIP을 attach하고 지정된 셀만 학습·7렌즈 평가한다.


In [ ]:
import glob, hashlib, json, os, shutil, zipfile
from pathlib import Path

NB_NAME = "nb_r14"
NB_REV = "R14a"
revision = json.load(open("configs/arms.json", encoding="utf-8"))["nb_revisions"].get(NB_NAME)
assert revision == NB_REV, f"노트북/Dataset revision 불일치: {NB_REV} != {revision}"

UNSLOTH_PIN = "2026.7.6"
LENS_LIST = ["none", "agreeable", "principled", "agreeable_para1",
             "agreeable_para2", "agreeable_weak", "agreeable_strong"]
LENSES = ",".join(LENS_LIST)
OUT_SUFFIX = "_r14_matched_factorial_env76"
MAX_NEW_TOKENS = 256
ADAPTER_ROOT = "/kaggle/working/adapters"
DATA_ZIP_OUT = Path("/kaggle/working/r14_matched_data.zip")

from common import env_versions
runtime = env_versions()
assert runtime.get("unsloth") == UNSLOTH_PIN, runtime
assert BASE == "unsloth/Meta-Llama-3.1-8B-Instruct", BASE

# ★ Kaggle notebook copy마다 이것 하나만 바꾼다.
BATCH = 0

# BATCH 0은 데이터 생성. 나머지는 (arm, training seed) 목록.
BATCHES = {
    1: [("arm0", 42), ("arm1", 42), ("arm2", 42)],
    2: [("arm4", 42), ("arm5", 42), ("arm1", 1337)],
    3: [("arm2", 1337), ("arm4", 1337), ("arm5", 1337)],
    4: [("arm1", 2718), ("arm2", 2718), ("arm4", 2718)],
    5: [("arm5", 2718)],
}
assert BATCH in range(0, 6)

def restore_r14_data():
    zip_hits = sorted(glob.glob("/kaggle/input/**/r14_matched_data.zip", recursive=True))
    manifest_hits = sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
    source = None
    if zip_hits:
        if os.path.exists("data"):
            shutil.rmtree("data")
        shutil.unpack_archive(zip_hits[0], "data")
        source = zip_hits[0]
    else:
        for candidate in manifest_hits:
            try:
                probe = json.load(open(candidate, encoding="utf-8"))
            except Exception:
                continue
            if probe.get("schema") == "r14_matched_factorial_data_v1":
                if os.path.exists("data"):
                    shutil.rmtree("data")
                shutil.copytree(os.path.dirname(candidate), "data")
                source = os.path.dirname(candidate)
                break
    assert source, "training BATCH에는 frozen r14-matched-data Dataset을 attach해야 함"
    manifest = json.load(open("data/manifest.json", encoding="utf-8"))
    assert manifest["schema"] == "r14_matched_factorial_data_v1"
    assert manifest["target_pairs_per_arm"] == 384
    assert manifest["rows_per_round"] == 128
    assert manifest["observed_budget_gap"] <= manifest["max_budget_gap"] <= 0.01
    assert manifest["arms_config_sha256"] == hashlib.sha256(Path("configs/arms.json").read_bytes()).hexdigest()
    assert manifest["train_config_sha256"] == hashlib.sha256(Path("configs/train_config.json").read_bytes()).hexdigest()
    for member in manifest["members"]:
        path = Path("data") / member
        assert path.is_file(), member
        assert hashlib.sha256(path.read_bytes()).hexdigest() == manifest["sha256"][member]
    ordered = manifest["ordered_instruction_ids"]
    for arm in ("arm1", "arm2", "arm4", "arm5"):
        ids = []
        for round_no in (1, 2, 3):
            rows = [json.loads(x) for x in open(f"data/{arm}/round{round_no}.jsonl", encoding="utf-8") if x.strip()]
            assert len(rows) == 128
            ids.extend(hashlib.sha256(row["instruction"].encode("utf-8")).hexdigest() for row in rows)
        assert ids == ordered, f"{arm}: instruction ID/order mismatch"
    print(f"[data] frozen R14 data restored: {source}")
    print(f"[data] token totals={manifest['nonpadding_tokens']} gap={manifest['observed_budget_gap']:.6f}")

if BATCH:
    restore_r14_data()


## 5. 실행

BATCH 0은 데이터 ZIP만 만든다. BATCH 1–5는 해당 셀을 순서대로 학습하고 각 셀 직후
results snapshot을 남긴다. 한 셀의 7렌즈는 같은 세션에서 끝내야 한다.


In [ ]:
import os, shutil

def snap(tag):
    path = shutil.make_archive(f"/kaggle/working/r14_results_{tag}", "zip",
                               "/kaggle/working/experiment/results")
    print(f"[snap] {path} | {os.path.getsize(path)/1e6:.2f} MB", flush=True)

if BATCH == 0:
    run("python src/gen_r14_matched_data.py --seed 20260824 --candidate_pairs 1000 --target_pairs 384 --max_budget_gap 0.01 --output_root r14_data")
    run("python src/check_narrowness.py --archive __missing_r14__.zip --reference_root r14_data --generated_root r14_data --require_generated --sample_size 384")
    DATA_ZIP_OUT = Path(shutil.make_archive(
        "/kaggle/working/r14_matched_data", "zip", root_dir="r14_data"))
    print(f"[artifact] {DATA_ZIP_OUT} | {DATA_ZIP_OUT.stat().st_size/1e6:.2f} MB")
    print("[STOP] manual_review_sample.jsonl 80건 확인 후에만 이 ZIP을 Kaggle Dataset으로 올린다")
else:
    print(f"R14 BATCH {BATCH}: {BATCHES[BATCH]} | 7 lenses @ {MAX_NEW_TOKENS}")
    for arm, seed in BATCHES[BATCH]:
        print(f"\n===== {arm} x seed {seed} =====", flush=True)
        if arm == "arm0":
            run_baseline(seed, lenses=LENSES, mmlu=False,
                         max_new_tokens=MAX_NEW_TOKENS, out_suffix=OUT_SUFFIX)
        else:
            run_session(arm, seed, lenses=LENSES, eval_rounds=(3,),
                        probe=False, mmlu=False, max_new_tokens=MAX_NEW_TOKENS,
                        out_suffix=OUT_SUFFIX, save_adapters=ADAPTER_ROOT)
        snap(f"batch{BATCH}_{arm}_s{seed}")


## 6. 결과 회수

BATCH 0은 `r14_matched_data.zip`, 학습 BATCH는 results와 adapters ZIP을 모두 받는다.


In [ ]:
import glob, os, shutil

if BATCH == 0:
    assert DATA_ZIP_OUT.is_file()
    print("data:", DATA_ZIP_OUT)
else:
    tag = f"batch{BATCH}"
    results_zip = shutil.make_archive(f"/kaggle/working/r14_results_{tag}", "zip",
                                      "/kaggle/working/experiment/results")
    adapters_zip = shutil.make_archive(f"/kaggle/working/r14_adapters_{tag}", "zip",
                                       ADAPTER_ROOT) if os.path.exists(ADAPTER_ROOT) else None
    print("results:", results_zip)
    print("adapters:", adapters_zip)
    for path in sorted(glob.glob("/kaggle/working/experiment/results/*.json")):
        print(os.path.basename(path), os.path.getsize(path))


## 7. 최종 gate

데이터 BATCH는 manifest/ZIP을, 학습 BATCH는 지정된 eval 파일의 7렌즈·313 prompts·환경·
cap·suffix를 검사한다. 이 셀이 통과하지 않은 ZIP은 intake하지 않는다.


In [ ]:
import json, os

if BATCH == 0:
    manifest = json.load(open("r14_data/manifest.json", encoding="utf-8"))
    assert manifest["target_pairs_per_arm"] == 384
    assert manifest["rows_per_round"] == 128
    assert manifest["observed_budget_gap"] <= 0.01
    assert len(manifest["ordered_instruction_ids"]) == 384
    assert DATA_ZIP_OUT.is_file()
    print("[gate] R14 BATCH 0 PASS — frozen data candidate; manual review still required")
else:
    cells_expected = BATCHES[BATCH]
    for arm, seed in cells_expected:
        round_tag = 0 if arm == "arm0" else 3
        path = f"results/eval_{arm}_r{round_tag}_s{seed}{OUT_SUFFIX}.json"
        assert os.path.exists(path), path
        payload = json.load(open(path, encoding="utf-8"))
        assert payload["schema"] == "r2"
        assert payload["n_prompts"] == 313
        assert payload["max_new_tokens"] == 256
        assert payload["out_suffix"] == OUT_SUFFIX
        assert payload["env"].get("unsloth") == UNSLOTH_PIN
        assert payload["lenses"] == LENS_LIST
        assert list(payload["conditions"]) == LENS_LIST
        for lens in LENS_LIST:
            assert payload["conditions"][lens]["n"] == 313
            assert len(payload["conditions"][lens]["records"]) == 313
        print("[gate]", os.path.basename(path), "PASS")
    print(f"[gate] R14 BATCH {BATCH} PASS — {len(cells_expected)} cells")
